# PolyAI Training on Google Colab (Tesla T4)

This notebook sets up and runs your Rust-based MCTS Zero training on Colab's GPU.

## 1. Check GPU

In [2]:
!nvidia-smi

Tue Feb  3 22:32:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Rust

In [3]:
# Install Rust
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ['PATH']}"
!rustc --version

info: downloading installer
info: profile set to 'default'
info: default host triple is x86_64-unknown-linux-gnu
info: syncing channel updates for 'stable-x86_64-unknown-linux-gnu'
info: latest update on 2026-01-22, rust version 1.93.0 (254b59607 2026-01-19)
info: downloading component 'cargo'
info: downloading component 'clippy'
info: downloading component 'rust-docs'
info: downloading component 'rust-std'
info: downloading component 'rustc'
info: downloading component 'rustfmt'
info: installing component 'cargo'
 10.3 MiB /  10.3 MiB (100 %)   8.7 MiB/s in  1s         
info: installing component 'clippy'
info: installing component 'rust-docs'
 20.7 MiB /  20.7 MiB (100 %)   4.8 MiB/s in  4s         
info: installing component 'rust-std'
 28.2 MiB /  28.2 MiB (100 %)  10.9 MiB/s in  3s         
info: installing component 'rustc'
 74.4 MiB /  74.4 MiB (100 %)  10.5 MiB/s in  7s         
info: installing component 'rustfmt'
info: default toolchain set to 'stable-x86_64-unknown-linux-gnu

## 3. Clone Your Repository

In [ ]:
# !git clone https://github.com/HenBOMB/Polyfish
!git fetch
!git pull
%cd Polyfish/polyfish-rs

/content/Polyfish/polyfish-rs


## 4. Build with CUDA Support

In [7]:
# Build in release mode with CUDA
!cargo build --release --features cuda --bin self_play

    Updating crates.io index
  Downloaded atomic-waker v1.1.2
  Downloaded gemm-f64 v0.18.2
  Downloaded bindgen_cuda v0.1.6
  Downloaded async-trait v0.1.89
  Downloaded autocfg v1.5.0
  Downloaded bitflags v2.10.0
  Downloaded num_enum_derive v0.7.5
  Downloaded rand v0.8.5
  Downloaded getrandom v0.2.17
  Downloaded glob v0.3.3
  Downloaded half v2.7.1
  Downloaded parking_lot v0.12.5
  Downloaded rand_core v0.9.5
  Downloaded zerofrom-derive v0.1.6
  Downloaded zerofrom v0.1.6
  Downloaded tokio-macros v2.6.0
  Downloaded thiserror v1.0.69
  Downloaded proc-macro2 v1.0.106
  Downloaded mime_guess v2.0.5
  Downloaded bytemuck_derive v1.10.2
  Downloaded sync_wrapper v1.0.2
  Downloaded hyper-util v0.1.19
  Downloaded ug v0.1.0
  Downloaded pulp v0.21.5
  Downloaded ug-cuda v0.1.0
  Downloaded seq-macro v0.3.6
  Downloaded zmij v1.0.17
  Downloaded zip v1.1.4
  Downloaded tracing-core v0.1.36
  Downloaded serde v1.0.228
  Downloaded zerocopy-derive v0.8.34
  Downloaded tower-http v0.

## 5. Run Benchmark

In [8]:
# Test GPU performance
!cargo run --release --features cuda --bin benchmark

   --> src/ai/mcts_zero.rs:263:13
    |
263 |         let mut current = root;
    |             ----^^^^^^^
    |             |
    |             help: remove this `mut`
    |
    = note: `#[warn(unused_mut)]` (part of `#[warn(unused)]`) on by default

  --> src/ai/mcts_zero.rs:44:8
   |
31 | impl ZeroNode {
   | ------------- methods in this implementation
...
44 |     fn value(&self) -> f32 {
   |        ^^^^^
...
91 |     fn select_child(&mut self, c_puct: f32) -> Option<&mut ZeroNode> {
   |        ^^^^^^^^^^^^
   |
   = note: `#[warn(dead_code)]` (part of `#[warn(unused)]`) on by default

   --> src/ai/mcts_zero.rs:455:8
    |
129 | impl<'a> ZeroMctsAgent<'a> {
    | -------------------------- method in this implementation
...
455 |     fn search(&self, game: &mut Game, node: &mut ZeroNode) -> f32 {
    |        ^^^^^^

   Compiling polyfish v0.1.0 (/content/Polyfish/polyfish-rs)
    Finished `release` profile [optimized] target(s) in 33.77s
     Running `target/release/benchmark`

## 6. Run Training

In [ ]:
# Set environment variables
os.environ['NUM_GAMES'] = '30'
os.environ['MCTS_ITERS'] = '100'

# Run self-play
!cargo run --release --bin self_play

## 7. Download Results

In [ ]:
# Download model and training data
from google.colab import files

# Download model
files.download('model.safetensors')

# Download training data
!zip -r training_data.zip game_*.json
files.download('training_data.zip')